# Extended Filter Set QHARM Free Energy

Compute missing quasi-harmonic free energies for the 946 transition states in `Data/TS/result_filter_extended.csv`. The TS log lookup matches `3_Build_DataBase.ipynb`: try the base name first, then the conformer-suffixed fallback. Existing component QHARM energies are read from `BorylXAT-DB_qh_update.db`.

The calculation is cached and resumable. Original RRHO columns are preserved; validated QHARM columns are appended to the CSV.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from DFTStructureGenerator.project_paths import REPO_ROOT, TS_DATA_DIR, raw_calc_path
from DFTStructureGenerator.thermochemistry import database_path, load_structure_thermochemistry

HARTREE_TO_KCAL = 627.509474
TEMPERATURE_K = 298.15
FREQUENCY_CUTOFF_CM = 100.0
INPUT_CSV = TS_DATA_DIR / 'result_filter_extended.csv'
QHARM_DB = database_path(True)
RAW_TS_DIR = raw_calc_path('Sum', 'TS_needIRC')
OUTDIR = REPO_ROOT / 'output' / 'jupyter-notebook'
CACHE_CSV = OUTDIR / 'extended_qharm_ts_thermochemistry.csv'
AUDIT_CSV = OUTDIR / 'extended_qharm_export_audit.csv'
BACKUP_CSV = REPO_ROOT / 'output' / 'qharm_csv_backups' / 'Data' / 'TS' / INPUT_CSV.name

FORCE_RECOMPUTE = False
RETRY_FAILURES = True
MAX_ROWS = None  # Set an integer only for a quick test.
CHECKPOINT_EVERY = 25
WRITE_IN_PLACE = True

OUTDIR.mkdir(parents=True, exist_ok=True)
for path in [INPUT_CSV, QHARM_DB, RAW_TS_DIR]:
    if not path.exists():
        raise FileNotFoundError(path)
print('Input:', INPUT_CSV)
print('Raw TS directory:', RAW_TS_DIR)
print('Cache:', CACHE_CSV)


## Build and validate the TS log manifest


In [ ]:
extended = pd.read_csv(INPUT_CSV)
id_columns = ['B_Index', 'N_Index', 'Cl_Index', 'Conf_id']
for column in id_columns:
    extended[column] = extended[column].astype(int)
if extended.duplicated(['B_Index', 'N_Index', 'Cl_Index']).any():
    raise ValueError('Extended set contains duplicate B/N/Cl reaction keys.')

manifest_rows = []
for row in extended.itertuples(index=False):
    stem = f'B_{row.B_Index:05d}_Nu_{row.N_Index:05d}_Cl_{row.Cl_Index:05d}'
    base = RAW_TS_DIR / f'{stem}.log'
    fallback = RAW_TS_DIR / f'{stem}_{row.Conf_id:04d}.log'
    selected = base if base.exists() else fallback
    manifest_rows.append({
        'key': f'B_{row.B_Index:05d}_LB_{row.N_Index:05d}_Cl_{row.Cl_Index:05d}',
        'B_Index': row.B_Index, 'N_Index': row.N_Index, 'Cl_Index': row.Cl_Index,
        'Conf_id': row.Conf_id, 'source_TS_G_hartree': float(row.TS_G),
        'base_log_exists': base.exists(), 'fallback_log_exists': fallback.exists(),
        'log_file': str(selected), 'log_exists': selected.exists(),
    })
manifest = pd.DataFrame(manifest_rows)
if not manifest['log_exists'].all():
    display(manifest.loc[~manifest['log_exists']].head(20))
    raise FileNotFoundError(f"Missing {int((~manifest['log_exists']).sum())} TS logs")
print(manifest[['base_log_exists', 'fallback_log_exists', 'log_exists']].sum())
display(manifest.head())


## AaronTools conversion

The definitions and 100 cm⁻¹ cutoff match `revision_quasi_harmonic_free_energy.ipynb`.


In [ ]:
try:
    import AaronTools
    from AaronTools.comp_output import CompOutput
    print('AaronTools:', getattr(AaronTools, '__version__', None), AaronTools.__file__)
except Exception as exc:
    raise ImportError('Run this notebook with the main_py3_12 environment containing AaronTools.') from exc

def compute_aarontools_thermo(log_file, temperature=TEMPERATURE_K, cutoff_cm=FREQUENCY_CUTOFF_CM):
    co = CompOutput(str(log_file))
    if co.frequency is None:
        raise ValueError('No frequency block found')
    if co.energy is None:
        raise ValueError('No electronic energy found')
    rrho_dg = co.calc_G_corr(temperature=temperature, v0=0, method='RRHO')
    qharm_dg = co.calc_G_corr(temperature=temperature, v0=cutoff_cm, method='QHARM')
    qrrho_dg = co.calc_G_corr(temperature=temperature, v0=cutoff_cm, method='QRRHO')
    _, full_dh, full_entropy = co.therm_corr(
        temperature=temperature, v0=cutoff_cm, method='QRRHO', enthalpy_method='QRRHO'
    )
    full_qrrho_dg = full_dh - temperature * full_entropy
    frequencies = np.asarray(co.frequency.real_frequencies, dtype=float)
    positive = frequencies[frequencies > 0]
    low = positive[positive < cutoff_cm]
    return {
        'electronic_energy_hartree': float(co.energy),
        'gibbs_rrho_hartree': float(co.energy + rrho_dg),
        'gibbs_qharm_hartree': float(co.energy + qharm_dg),
        'gibbs_qrrho_hartree': float(co.energy + qrrho_dg),
        'gibbs_full_qrrho_hartree': float(co.energy + full_qrrho_dg),
        'rrho_g_corr_hartree': float(rrho_dg),
        'qharm_g_corr_hartree': float(qharm_dg),
        'temperature_K': float(temperature),
        'frequency_cutoff_cm_1': float(cutoff_cm),
        'n_real_frequencies': int(len(frequencies)),
        'n_low_positive_frequencies': int(len(low)),
        'min_positive_frequency_cm_1': float(positive.min()) if len(positive) else np.nan,
        'low_positive_frequencies_cm_1': json.dumps(low.tolist()),
    }


In [ ]:
sample = manifest.iloc[0]
t0 = time.perf_counter()
sample_result = compute_aarontools_thermo(sample['log_file'])
sample_elapsed = time.perf_counter() - t0
print(sample[['key', 'log_file']].to_string())
print(f'Sample elapsed: {sample_elapsed:.3f} s')
display(pd.Series(sample_result).drop(labels='low_positive_frequencies_cm_1').to_frame('value'))


## Compute missing TS rows with checkpoints


In [ ]:
if CACHE_CSV.exists() and not FORCE_RECOMPUTE:
    cached = pd.read_csv(CACHE_CSV)
else:
    cached = pd.DataFrame()

completed_keys = set()
if not cached.empty:
    completed_keys = set(cached.loc[cached['status'].eq('ok'), 'key'])
    if not RETRY_FAILURES:
        completed_keys.update(cached.loc[cached['status'].ne('ok'), 'key'])
pending = manifest.loc[~manifest['key'].isin(completed_keys)].copy()
if MAX_ROWS is not None:
    pending = pending.head(int(MAX_ROWS))
print(f'Cached rows: {len(cached)}; pending rows: {len(pending)}')

records = cached.to_dict('records') if not cached.empty else []
run_start = time.perf_counter()
for sequence, rec in enumerate(pending.to_dict('records'), start=1):
    row = {**rec, 'status': 'ok', 'error_type': None, 'error': None}
    started = time.perf_counter()
    try:
        row.update(compute_aarontools_thermo(rec['log_file']))
    except Exception as exc:
        row.update({
            'status': 'error', 'error_type': type(exc).__name__, 'error': str(exc),
            'traceback': traceback.format_exc(limit=5),
        })
    row['elapsed_seconds'] = time.perf_counter() - started
    records.append(row)
    if sequence % CHECKPOINT_EVERY == 0 or sequence == len(pending):
        cache = pd.DataFrame(records).drop_duplicates('key', keep='last').sort_values('key')
        cache.to_csv(CACHE_CSV, index=False)
        print(f'Checkpoint {sequence}/{len(pending)}; elapsed {time.perf_counter() - run_start:.1f} s')

thermo_cache = pd.read_csv(CACHE_CSV) if CACHE_CSV.exists() else pd.DataFrame(records)
display(thermo_cache['status'].value_counts(dropna=False).to_frame('rows'))
if len(thermo_cache) != len(manifest) and MAX_ROWS is None:
    raise ValueError(f'Cache coverage mismatch: {len(thermo_cache)} != {len(manifest)}')


## Combine TS and component QHARM energies


In [ ]:
structure_thermo = load_structure_thermochemistry(QHARM_DB)
g_qharm = structure_thermo.set_index('key')['gibbs_qharm_hartree'].to_dict()
result = extended.drop(columns=[
    column for column in [
        'G_energy_qharm', 'TS_G_qharm', 'deltaG_react_qharm',
        'deltaG_qharm(kcal/mol)', 'deltaGa_qharm(kcal/mol)',
    ] if column in extended.columns
]).copy()
ok_ts = thermo_cache.loc[thermo_cache['status'].eq('ok'), [
    'B_Index', 'N_Index', 'Cl_Index', 'Conf_id', 'gibbs_qharm_hartree'
]].rename(columns={'gibbs_qharm_hartree': 'TS_G_qharm'})
result = result.merge(ok_ts, on=id_columns, how='left', sort=False, validate='one_to_one')

reactant_qh = []
product_qh = []
missing_component_keys = []
for row in result.itertuples(index=False):
    b, n, c = int(row.B_Index), int(row.N_Index), int(row.Cl_Index)
    keys = {
        'bn_r': f'B_{b:05d}_LB_{n:05d}_r', 'bn_p': f'B_{b:05d}_LB_{n:05d}_p',
        'cl_r': f'Cl_{c:05d}_r', 'cl_p': f'Cl_{c:05d}_p',
    }
    missing = [key for key in keys.values() if key not in g_qharm]
    if missing:
        missing_component_keys.extend(missing)
        reactant_qh.append(np.nan); product_qh.append(np.nan)
    else:
        reactant_qh.append(g_qharm[keys['bn_r']] + g_qharm[keys['cl_r']])
        product_qh.append(g_qharm[keys['bn_p']] + g_qharm[keys['cl_p']])
if missing_component_keys:
    raise KeyError(f'Missing component QHARM keys: {sorted(set(missing_component_keys))[:10]}')
result['G_energy_qharm'] = reactant_qh
result['deltaG_react_qharm'] = (np.asarray(product_qh) - result['G_energy_qharm']) * HARTREE_TO_KCAL
result['deltaG_qharm(kcal/mol)'] = result['deltaG_react_qharm']
result['deltaGa_qharm(kcal/mol)'] = (result['TS_G_qharm'] - result['G_energy_qharm']) * HARTREE_TO_KCAL

qharm_columns = [
    'G_energy_qharm', 'TS_G_qharm', 'deltaG_react_qharm',
    'deltaG_qharm(kcal/mol)', 'deltaGa_qharm(kcal/mol)',
]
if result[qharm_columns].isna().any().any():
    raise ValueError(f'Incomplete output columns: {result[qharm_columns].isna().sum().to_dict()}')
original_columns = [column for column in extended.columns if column not in qharm_columns]
if not result[original_columns].equals(extended[original_columns]):
    raise ValueError('Original CSV columns or row order changed.')
result = result[original_columns + qharm_columns]

shift_summary = pd.DataFrame({
    'quantity': ['barrier', 'reaction_energy'],
    'mean_shift_kcal_mol': [
        (result['deltaGa_qharm(kcal/mol)'] - result['deltaGa(kcal/mol)']).mean(),
        (result['deltaG_qharm(kcal/mol)'] - result['deltaG(kcal/mol)']).mean(),
    ],
    'max_abs_shift_kcal_mol': [
        (result['deltaGa_qharm(kcal/mol)'] - result['deltaGa(kcal/mol)']).abs().max(),
        (result['deltaG_qharm(kcal/mol)'] - result['deltaG(kcal/mol)']).abs().max(),
    ],
})
display(shift_summary)
display(result[qharm_columns].describe())


## Write the enriched extended CSV


In [ ]:
audit = {
    'input_csv': str(INPUT_CSV), 'rows': len(result),
    'successful_ts_qharm': int(thermo_cache['status'].eq('ok').sum()),
    'failed_ts_qharm': int(thermo_cache['status'].ne('ok').sum()),
    'temperature_K': TEMPERATURE_K, 'frequency_cutoff_cm_1': FREQUENCY_CUTOFF_CM,
    'mean_barrier_shift_kcal_mol': float((result['deltaGa_qharm(kcal/mol)'] - result['deltaGa(kcal/mol)']).mean()),
    'mean_reaction_shift_kcal_mol': float((result['deltaG_qharm(kcal/mol)'] - result['deltaG(kcal/mol)']).mean()),
}
pd.DataFrame([audit]).to_csv(AUDIT_CSV, index=False)

if WRITE_IN_PLACE:
    BACKUP_CSV.parent.mkdir(parents=True, exist_ok=True)
    if not BACKUP_CSV.exists():
        shutil.copy2(INPUT_CSV, BACKUP_CSV)
    temporary = INPUT_CSV.with_name(INPUT_CSV.name + '.qharm.tmp')
    result.to_csv(temporary, index=False)
    os.replace(temporary, INPUT_CSV)
    print('Updated:', INPUT_CSV)
else:
    print('Validation-only run: source CSV unchanged.')
print('Audit:', AUDIT_CSV)
print(json.dumps(audit, indent=2))
